In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # this loads .env file

client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPEN_AI_KEY"),
)

response = client.responses.create(
    model="gpt-5.2",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)

Arrr, ye use `isinstance()` to see if an object be belongin’ to a class (or a crew of classes).

```python
isinstance(obj, MyClass)
```

Example:
```python
class Pirate:
    pass

jack = Pirate()

print(isinstance(jack, Pirate))  # True
print(isinstance("rum", Pirate)) # False
```

If ye want to check against multiple classes:
```python
isinstance(obj, (ClassA, ClassB))
```

And note, matey: `isinstance()` also returns `True` for subclasses, which be usually what ye want.


In [5]:
chat_completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Hello World"}]
)

In [6]:
chat_completion.choices[0].message.content

'Hello! How can I assist you today?'

In [42]:
class Agent:
    """
    Intialize the agent with the system prompt.
    Add messages queue to keep track of past results and prompts.
    """
    def __init__(self, system_prompt=""):
        self.system_prompt = system_prompt
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                model="gpt-4o",
                temperature=0,
                messages=self.messages
        )
        return completion.choices[0].message.content

In [29]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns a number - uses Python so be sure to use floating point 
syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given a breed

Example session:
Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A bulldog weighs 51 lbs

You then
output:
Answer: A bullgog weighs 51 lbs.
""".strip()

In [14]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")    

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [33]:
a_bot = Agent(system_prompt)

In [34]:
result = a_bot("How much does a toy poodle weigh?")

OpenAI content:  Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight action.
Action: average_dog_weight: Toy Poodle
PAUSE


In [35]:
result

'Thought: I should look up the average weight of a Toy Poodle using the average_dog_weight action.\nAction: average_dog_weight: Toy Poodle\nPAUSE'

In [36]:
result = average_dog_weight("Toy Poodle")

In [37]:
result

'a toy poodles average weight is 7 lbs'

In [38]:
next_prompt = "Observation: {}".format(result)

In [39]:
a_bot(next_prompt)

OpenAI content:  Answer: A Toy Poodle weighs on average 7 lbs.


'Answer: A Toy Poodle weighs on average 7 lbs.'

In [40]:
a_bot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns a number - uses Python so be sure to use floating point \nsyntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given a breed\n\nExample session:\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A bulldog weighs 51 lbs\n\nYou then\noutput:\nAnswer: A bullgog weighs 51 lbs.'},
 {'role': 'user', 'content': 'How much does a toy poo

In [43]:
sec_bot = Agent(system_prompt)

In [44]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
sec_bot(question)

"Thought: I need to find the average weight of a Border Collie and a Scottish Terrier, then add them together to get the combined weight. I'll start by finding the weight of a Border Collie. \nAction: average_dog_weight: Border Collie\nPAUSE"

In [54]:
result = average_dog_weight("Border Collie")

In [55]:
result

'a Border Collies average weight is 37 lbs'

In [46]:
sec_bot(f"Observation: {result}")

'Thought: Now that I have the average weight of a Border Collie, I need to find the average weight of a Scottish Terrier.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [47]:
result = average_dog_weight("Scottish Terrier")

In [48]:
sec_bot(f"Observation: {result}")

'Thought: I now have the average weights of both dogs. I will add the weight of the Border Collie (37 lbs) and the Scottish Terrier (20 lbs) to find their combined weight.\nAction: calculate: 37 + 20\nPAUSE'

In [50]:
final_result = calculate("37 + 20")

In [51]:
final_result

57

In [52]:
sec_bot(f"Observation: {final_result}")

'Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.'

In [53]:
sec_bot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns a number - uses Python so be sure to use floating point \nsyntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given a breed\n\nExample session:\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A bulldog weighs 51 lbs\n\nYou then\noutput:\nAnswer: A bullgog weighs 51 lbs.'},
 {'role': 'user',
  'content': 'I have 2 dogs, a bord